Global imports

In [1]:
import sys
import re
import os
import cv2
import pandas as pd
import seaborn as sns
import numpy as np
import ipywidgets as widgets
import matplotlib.pyplot as plt
import glob
import random
from pathlib import Path

import torch
from torch import nn

from torch.utils.data import DataLoader
from torch.utils.data import Dataset

from torchvision import datasets, transforms
from torchvision.utils import make_grid

# Note: this notebook requires torch >= 1.10.0
torch.__version__

'2.12.1+cu130'

Local imports

In [2]:
from src.data.plots import plot_class_distribution
from src.data.loaders import read_train_test_data_txt

In [3]:
# Setup device-agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

1. Configure Base Path

In [4]:
ROOT_DIR = Path.cwd().parent.parent
sys.path.insert(0, str(ROOT_DIR / "src"))

print(ROOT_DIR)

/home/samuel-linux/master-degree-project


In [5]:
TRAIN_TEST_DATA = ROOT_DIR / "data" / "UCF_Crime" / "raw"
TRAIN_TEST_DATA

PosixPath('/home/samuel-linux/master-degree-project/data/UCF_Crime/raw')

In [6]:
TRAIN_TEST_SPLIT_FILES = ROOT_DIR / "data" / "UCF_Crime" / "processed" / "train_test_split"
TRAIN_TEST_SPLIT_FILES

PosixPath('/home/samuel-linux/master-degree-project/data/UCF_Crime/processed/train_test_split')

In [7]:
train_files = read_train_test_data_txt(TRAIN_TEST_SPLIT_FILES, "Anomaly_Train")
test_files = read_train_test_data_txt(TRAIN_TEST_SPLIT_FILES, "Anomaly_Test")

2. Create Dataset

In [8]:
class CustomVideoDataset(Dataset):
  def __init__(self, root_dir, data_files, num_frames=16, transform=None):
    """
    Args:
        root_dir (str): Path to the dataset directory (e.g., 'my_video_dataset/training')
        num_frames (int): Number of frames to sample from each video.
        transform (callable, optional): PyTorch transforms to apply to each frame.
    """
    self.data_files = data_files
    self.root_dir = root_dir
    self.num_frames = num_frames
    self.transform = transform
      
    # Get class names and map them to integers
    # self.classes = sorted(os.listdir(root_dir))
    self.classes = list(dict.fromkeys(file.split("/")[0] for file in train_files))
    
    # self.idx_to_class = {i: cls_name for cls_name, i in self.classes}
    self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
      
    # Gather all video file paths and labels
    self.video_samples = []
    
    for data in self.data_files:
      class_name = data.split("/")[0]
      file_name = data.split("/")[1]
      if file_name.lower().endswith(('.mp4', '.avi', '.mov', '.mkv')):
        self.video_samples.append((os.path.join(TRAIN_TEST_DATA, class_name, file_name), self.class_to_idx[class_name]))
    
    # for cls_name in self.classes:
    #   cls_dir = os.path.join(root_dir, cls_name)
    #   if os.path.isdir(cls_dir):
    #     for file_name in os.listdir(cls_dir):
    #       if file_name.lower().endswith(('.mp4', '.avi', '.mov', '.mkv')):
    #         self.video_samples.append((os.path.join(cls_dir, file_name), self.class_to_idx[cls_name]))

  def __len__(self):
    return len(self.video_samples)

  def __getitem__(self, idx):
    video_path, label = self.video_samples[idx]
    
    # Capture frames from video
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if total_frames <= 0:
      raise ValueError(f"Error loading video or empty video file: {video_path}")
        
    # Calculate uniformly spaced frame indices
    seg_size = max(1, total_frames // self.num_frames)
    frame_indices = [min(total_frames - 1, i * seg_size) for i in range(self.num_frames)]
    
    frames = []
    for f_idx in frame_indices:
      cap.set(cv2.CAP_PROP_POS_FRAMES, f_idx)
      success, frame = cap.read()
      if not success:
        # Fallback to an empty/black frame if reading fails midway
        frame = torch.zeros((3, 224, 224)) if not self.transform else None
      else:
        # Convert BGR (OpenCV default) to RGB
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        if self.transform:
          frame = self.transform(frame)
              
      frames.append(frame)
        
    cap.release()
    
    # Stack frames into a single tensor
    # If your model expects shape (C, T, H, W) for 3D CNNs like ResNet3D:
    # permute channels after stacking.
    video_tensor = torch.stack(frames) # Shape: (T, C, H, W)
    video_tensor = video_tensor.permute(1, 0, 2, 3) # Shape: (C, T, H, W)
    
    return video_tensor, torch.tensor(label, dtype=torch.long)

In [9]:
# 1. Define Image/Frame transforms
# Video classification models typically require ImageNet or Kinetics normalization scales.
video_transforms = transforms.Compose([
  transforms.ToPILImage(),
  transforms.Resize((224, 224)),
  transforms.ToTensor(),
  transforms.Normalize(mean=[0.432, 0.394, 0.376], std=[0.228, 0.221, 0.216])
])

In [19]:
# 2. Instantiate Dataset
train_dataset = CustomVideoDataset(
  root_dir=TRAIN_TEST_DATA,
  data_files=train_files, 
  num_frames=32, 
  transform=video_transforms
)

val_dataset = CustomVideoDataset(
  root_dir=TRAIN_TEST_DATA,
  data_files=test_files, 
  num_frames=32, 
  transform=video_transforms
)

3. Create Dataloader

In [20]:
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

In [21]:
video, label = next(iter(train_dataloader))

ValueError: Caught ValueError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/home/samuel-linux/master-degree-project/.venv/lib/python3.12/site-packages/torch/utils/data/_utils/worker.py", line 374, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/home/samuel-linux/master-degree-project/.venv/lib/python3.12/site-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/tmp/ipykernel_12587/4210040232.py", line 48, in __getitem__
    raise ValueError(f"Error loading video or empty video file: {video_path}")
ValueError: Error loading video or empty video file: /home/samuel-linux/master-degree-project/data/UCF_Crime/raw/Explosion/Explosion041_x264.mp4


In [ ]:
# train shape
video.shape

4. Define Model

In [ ]:
class CNN3D(nn.Module):
  def __init__(self):
    super(CNN3D, self).__init__()
    
    self.conv3d = nn.Sequential(
      nn.Conv3d(in_channels=3, out_channels=64, kernel_size=(3, 3, 3), stride=1, padding=1),
      nn.ReLU(),
      nn.MaxPool3d((1, 2, 2)),

      nn.Conv3d(in_channels=64, out_channels=128, kernel_size=(3, 3, 3), stride=1, padding=1),
      nn.ReLU(),
      nn.MaxPool3d((2, 2, 2))
    )

  def forward(self, x):
    return self.conv3d(x)  # Shape: (Batch, 128, T//2, H//4, W//4)